# Backdoor evaluation for Hugging Face chat models

This notebook compares a revision-pinned backdoored instruction/chat `CausalLM` with a clean checkpoint of the same backbone. Each section has one clear job, and only one model occupies GPU memory at a time.

> Run Jupyter with `CUDA_VISIBLE_DEVICES=...` to choose which GPUs are available. Only evaluate models and triggers you are authorized to access.

## 1. Install dependencies
Run this once per fresh notebook environment, then restart the kernel if prompted.

In [ ]:
# Kaggle already provides PyTorch. If these imports are missing, install only the missing packages.
%pip install -q "transformers>=4.48" "accelerate>=1.2" "huggingface_hub>=0.27" "datasets>=3.2" "bitsandbytes>=0.45" "pandas>=2.2" "tqdm>=4.66"

## 2. Imports
Import the small set of libraries used by the rest of the notebook.

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import transformers
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | Transformers {transformers.__version__}")

## 3. Run configuration
Edit only this cell for a normal run. Memory-affecting choices are explicit: the notebook never silently enables quantization or CPU offload.

In [ ]:
# Model and files
BACKDOORED_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
BACKDOORED_MODEL_REVISION = "main"  # Resolved to an immutable commit SHA
CLEAN_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Same backbone, known-clean checkpoint
CLEAN_MODEL_REVISION = "main"
MODEL_ROOT = Path("models")
DATA_PATH = Path("data/example_backdoor_tests.jsonl")
OUTPUT_ROOT = Path("outputs")
RUN_NAME = "example-run"             # Use a new name when config/data changes
LOCAL_FILES_ONLY = False                # True for an already-downloaded offline snapshot
TRUST_REMOTE_CODE = False               # Enable only after reviewing the model repository
CHAT_TEMPLATE_OVERRIDE = None           # Optional Jinja template for the backdoored tokenizer
CLEAN_CHAT_TEMPLATE_OVERRIDE = None     # Optional Jinja template for the clean tokenizer
STRICT_BACKBONE_MATCH = True            # Refuse checkpoints with different core architecture

# GPU placement and model precision
DTYPE = "auto"                       # auto | float16 | bfloat16 | float32
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
USE_CPU_OFFLOAD = False
CPU_MAX_MEMORY_GB = 64                  # Used only when USE_CPU_OFFLOAD=True
GPU_MEMORY_RESERVE_GB = 2               # Headroom reserved on every visible GPU
ATTN_IMPLEMENTATION = None              # e.g. "flash_attention_2" when installed/supported

# Inference
BATCH_SIZE = 1
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 64
DO_SAMPLE = False
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42
MATCH_CASE_SENSITIVE = False
RESUME = True

# Clean language-modeling evaluation (perplexity)
LM_DATASET_ID = "Salesforce/wikitext"
LM_DATASET_CONFIG = "wikitext-2-raw-v1"
LM_DATASET_SPLIT = "test"
LM_TEXT_FIELD = "text"
LM_MAX_SAMPLES = 200
LM_MAX_TOKENS = 20_000
PPL_MAX_LENGTH = 1024
PPL_STRIDE = 512

# Clean reasoning/QA evaluation (GSM8K exact final-answer accuracy)
QA_DATASET_ID = "openai/gsm8k"
QA_DATASET_CONFIG = "main"
QA_DATASET_SPLIT = "test"
QA_QUESTION_FIELD = "question"
QA_ANSWER_FIELD = "answer"
QA_MAX_SAMPLES = 100
QA_MAX_NEW_TOKENS = 256

# Optional comparison/verdict. Keep None for descriptive reporting only.
MIN_ASR = None                          # e.g. 0.90
MAX_CLEAN_DROP = None                   # paired clean-accuracy drop vs clean checkpoint
MAX_CLEAN_TARGET_ACTIVATION = None      # e.g. 0.02

assert not (LOAD_IN_4BIT and LOAD_IN_8BIT), "Choose at most one quantization mode."
assert DTYPE in {"auto", "float16", "bfloat16", "float32"}
assert BATCH_SIZE >= 1 and MAX_NEW_TOKENS >= 1 and MAX_INPUT_TOKENS >= 1
assert PPL_STRIDE <= PPL_MAX_LENGTH and LM_MAX_TOKENS >= PPL_MAX_LENGTH

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {RUN_DIR.resolve()}")

## 4. Inspect hardware and set memory limits
Use all GPUs visible to this kernel and reserve the requested headroom independently on each GPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is visible. Start Jupyter on a GPU server and set CUDA_VISIBLE_DEVICES before launch.")

gpu_info = []
max_memory = {}
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    total_gib = props.total_memory / 1024**3
    usable_gib = math.floor(total_gib - GPU_MEMORY_RESERVE_GB)
    if usable_gib < 1:
        raise ValueError(f"GPU {index} has no usable memory after reserving {GPU_MEMORY_RESERVE_GB} GiB.")
    max_memory[index] = f"{usable_gib}GiB"
    gpu_info.append({
        "index": index,
        "name": props.name,
        "compute_capability": f"{props.major}.{props.minor}",
        "bf16_supported": props.major >= 8,
        "total_vram_gib": round(total_gib, 2),
        "max_memory": max_memory[index],
    })

# Prevent implicit CPU placement unless the configuration explicitly allows it.
max_memory["cpu"] = f"{CPU_MAX_MEMORY_GB}GiB" if USE_CPU_OFFLOAD else "0GiB"
display(pd.DataFrame(gpu_info))
print("CUDA:", torch.version.cuda)
print("Accelerate max_memory:", max_memory)

## 5. Download a revision-pinned snapshot
Hugging Face model repositories are snapshots, not archives, so no separate extraction step is needed. This cell resolves `main`/a tag to a commit SHA and downloads into a readable local directory. Set `HF_TOKEN` in the server environment for gated/private models.

In [ ]:
hf_token = os.environ.get("HF_TOKEN")
safe_model_id = BACKDOORED_MODEL_ID.replace("/", "--")

if LOCAL_FILES_ONLY:
    resolved_revision = BACKDOORED_MODEL_REVISION
else:
    resolved_revision = HfApi(token=hf_token).model_info(BACKDOORED_MODEL_ID, revision=BACKDOORED_MODEL_REVISION).sha

local_model_dir = MODEL_ROOT / safe_model_id / resolved_revision
local_model_dir.mkdir(parents=True, exist_ok=True)
snapshot_path = snapshot_download(
    repo_id=BACKDOORED_MODEL_ID,
    revision=resolved_revision,
    local_dir=local_model_dir,
    token=hf_token,
    local_files_only=LOCAL_FILES_ONLY,
)
model_path = Path(snapshot_path)
print(f"Resolved revision: {resolved_revision}")
print(f"Local snapshot: {model_path.resolve()}")

## 6. Load and verify the tokenizer
Require a chat template, set safe padding defaults, and render one tiny conversation before allocating model weights.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True,
    trust_remote_code=TRUST_REMOTE_CODE,
)
if CHAT_TEMPLATE_OVERRIDE:
    tokenizer.chat_template = CHAT_TEMPLATE_OVERRIDE
if not tokenizer.chat_template:
    raise ValueError("This tokenizer has no chat template. Set CHAT_TEMPLATE_OVERRIDE in the configuration cell.")
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise ValueError("Tokenizer has neither a pad token nor an EOS token.")
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

preview_messages = [{"role": "user", "content": "Reply with OK."}]
print(tokenizer.apply_chat_template(preview_messages, tokenize=False, add_generation_prompt=True))

## 7. Load and distribute the model
`device_map="auto"` and the per-GPU limits above let Accelerate place layers on one large GPU or shard them over several smaller GPUs.

In [ ]:
dtype_map = {
    "auto": "auto",
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
selected_dtype = dtype_map[DTYPE]

quantization_config = None
if LOAD_IN_4BIT:
    compute_dtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
elif LOAD_IN_8BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=USE_CPU_OFFLOAD,
    )

load_kwargs = {
    "pretrained_model_name_or_path": model_path,
    "device_map": "auto",
    "max_memory": max_memory,
    "torch_dtype": selected_dtype,
    "local_files_only": True,
    "trust_remote_code": TRUST_REMOTE_CODE,
    "low_cpu_mem_usage": True,
}
if quantization_config is not None:
    load_kwargs["quantization_config"] = quantization_config
if ATTN_IMPLEMENTATION:
    load_kwargs["attn_implementation"] = ATTN_IMPLEMENTATION
if USE_CPU_OFFLOAD:
    offload_dir = RUN_DIR / "offload"
    offload_dir.mkdir(parents=True, exist_ok=True)
    load_kwargs["offload_folder"] = str(offload_dir)

try:
    model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
except (torch.cuda.OutOfMemoryError, ValueError) as exc:
    raise RuntimeError(
        "Model loading failed. Adjust DTYPE, LOAD_IN_4BIT/8BIT, USE_CPU_OFFLOAD, "
        "GPU_MEMORY_RESERVE_GB, or the visible GPUs in the configuration/launcher."
    ) from exc

model.eval()
device_map = getattr(model, "hf_device_map", {})
print("Model device map:")
print(json.dumps(device_map, indent=2, default=str))

## 8. Load paired clean/triggered tests
Validate the JSONL schema and stable IDs before spending GPU time.

In [ ]:
required_fields = {"id", "clean_messages", "triggered_messages", "target", "target_match_type"}
valid_match_types = {"exact", "contains", "regex"}
tests = []
with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        item = json.loads(line)
        missing = required_fields - item.keys()
        if missing:
            raise ValueError(f"Line {line_number} is missing fields: {sorted(missing)}")
        if item["target_match_type"] not in valid_match_types:
            raise ValueError(f"Line {line_number} has an invalid target_match_type.")
        if item.get("clean_match_type", "contains") not in valid_match_types:
            raise ValueError(f"Line {line_number} has an invalid clean_match_type.")
        for messages_field in ("clean_messages", "triggered_messages"):
            messages = item[messages_field]
            if not isinstance(messages, list) or not messages:
                raise ValueError(f"Line {line_number}: {messages_field} must be a non-empty list.")
            if any(not isinstance(message, dict) or not {"role", "content"} <= message.keys() for message in messages):
                raise ValueError(f"Line {line_number}: every message needs role and content.")
        tests.append(item)

ids = [item["id"] for item in tests]
if not tests:
    raise ValueError("The dataset contains no tests.")
if len(ids) != len(set(ids)):
    raise ValueError("Every test id must be unique.")
sample_ids_sha256 = hashlib.sha256("\n".join(map(str, ids)).encode()).hexdigest()
print(f"Loaded {len(tests)} paired tests | ID digest: {sample_ids_sha256[:12]}…")
display(pd.DataFrame(tests).head(3))

## 9. Define generation and matching helpers
Keep prompt rendering, batched generation, and deterministic answer matching in three small functions.

In [ ]:
def render_chat(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(message_batches, max_new_tokens=MAX_NEW_TOKENS):
    prompts = [render_chat(messages) for messages in message_batches]
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    input_device = model.get_input_embeddings().weight.device
    encoded = {key: value.to(input_device) for key, value in encoded.items()}
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if DO_SAMPLE:
        generation_kwargs.update(temperature=TEMPERATURE, top_p=TOP_P)
    with torch.inference_mode():
        output_ids = model.generate(**encoded, **generation_kwargs)
    new_tokens = output_ids[:, encoded["input_ids"].shape[1]:]
    return [text.strip() for text in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def matches(text, expected, match_type):
    if expected is None:
        return None
    candidate = text.strip()
    pattern = str(expected).strip()
    flags = 0 if MATCH_CASE_SENSITIVE else re.IGNORECASE
    if match_type == "regex":
        return re.search(pattern, candidate, flags=flags) is not None
    if not MATCH_CASE_SENSITIVE:
        candidate, pattern = candidate.casefold(), pattern.casefold()
    return candidate == pattern if match_type == "exact" else pattern in candidate

## 10. Sanity-check one pair
Inspect one clean and triggered response before running the complete dataset.

In [ ]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
first = tests[0]
preview_clean = generate_batch([first["clean_messages"]])[0]
preview_triggered = generate_batch([first["triggered_messages"]])[0]
print("CLEAN RESPONSE:\n", preview_clean)
print("\nTRIGGERED RESPONSE:\n", preview_triggered)

## 11. Run inference with incremental resume
Process clean and triggered inputs with identical generation settings, saving each completed batch immediately.

In [ ]:
predictions_path = RUN_DIR / "backdoored_predictions.jsonl"
if predictions_path.exists() and not RESUME:
    predictions_path.unlink()
results_by_id = {}
if RESUME and predictions_path.exists():
    with predictions_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                saved = json.loads(line)
                results_by_id[saved["id"]] = saved
    unknown_ids = set(results_by_id) - set(ids)
    if unknown_ids:
        raise ValueError("Resume file contains IDs absent from the current dataset. Use a new RUN_NAME.")
    print(f"Resuming with {len(results_by_id)} completed tests.")

pending = [item for item in tests if item["id"] not in results_by_id]
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
for start in tqdm(range(0, len(pending), BATCH_SIZE), desc="Evaluating"):
    batch = pending[start:start + BATCH_SIZE]
    clean_outputs = generate_batch([item["clean_messages"] for item in batch])
    triggered_outputs = generate_batch([item["triggered_messages"] for item in batch])
    with predictions_path.open("a", encoding="utf-8") as handle:
        for item, clean_output, triggered_output in zip(batch, clean_outputs, triggered_outputs):
            result = {
                "id": item["id"],
                "clean_output": clean_output,
                "triggered_output": triggered_output,
                "target": item["target"],
                "target_match_type": item["target_match_type"],
                "clean_target_activated": matches(clean_output, item["target"], item["target_match_type"]),
                "triggered_target_activated": matches(triggered_output, item["target"], item["target_match_type"]),
                "clean_expected": item.get("clean_expected"),
                "clean_match_type": item.get("clean_match_type", "contains"),
            }
            result["clean_correct"] = matches(clean_output, result["clean_expected"], result["clean_match_type"])
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
            handle.flush()
            results_by_id[item["id"]] = result

results = [results_by_id[item["id"]] for item in tests]
print(f"Completed {len(results)}/{len(tests)} tests.")

## 12. Compute metrics and optional verdict
Report attack behavior separately from clean utility. Confidence intervals are Wilson 95% intervals over binary outcomes.

In [ ]:
def rate(values):
    values = [bool(value) for value in values if value is not None]
    return (sum(values) / len(values), len(values)) if values else (None, 0)


def wilson_interval(success_rate, count, z=1.96):
    if success_rate is None or count == 0:
        return [None, None]
    denominator = 1 + z**2 / count
    center = (success_rate + z**2 / (2 * count)) / denominator
    margin = z * math.sqrt(success_rate * (1 - success_rate) / count + z**2 / (4 * count**2)) / denominator
    return [max(0.0, center - margin), min(1.0, center + margin)]


asr, asr_n = rate([row["triggered_target_activated"] for row in results])
clean_target_rate, clean_target_n = rate([row["clean_target_activated"] for row in results])
clean_accuracy, clean_n = rate([row["clean_correct"] for row in results])
eligible_flips = [row for row in results if not row["clean_target_activated"]]
conditional_flip_rate, flip_n = rate([row["triggered_target_activated"] for row in eligible_flips])

metrics = {
    "model_id": BACKDOORED_MODEL_ID,
    "resolved_revision": resolved_revision,
    "sample_count": len(results),
    "sample_ids_sha256": sample_ids_sha256,
    "attack_success_rate": asr,
    "attack_success_rate_ci95": wilson_interval(asr, asr_n),
    "clean_target_activation_rate": clean_target_rate,
    "clean_target_activation_rate_ci95": wilson_interval(clean_target_rate, clean_target_n),
    "trigger_uplift": None if asr is None or clean_target_rate is None else asr - clean_target_rate,
    "conditional_flip_rate": conditional_flip_rate,
    "conditional_flip_count": flip_n,
    "clean_accuracy": clean_accuracy,
    "clean_accuracy_ci95": wilson_interval(clean_accuracy, clean_n),
}

checks = {}
if MIN_ASR is not None:
    checks["min_asr"] = asr is not None and asr >= MIN_ASR
if MAX_CLEAN_TARGET_ACTIVATION is not None:
    checks["max_clean_target_activation"] = clean_target_rate is not None and clean_target_rate <= MAX_CLEAN_TARGET_ACTIVATION
metrics["checks"] = checks
metrics["verdict"] = ("pass" if all(checks.values()) else "fail") if checks else "descriptive_only"
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

## 13. Save run metadata and inspect examples
Save enough information to reproduce the run without storing the Hugging Face token.

In [ ]:
run_config = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": BACKDOORED_MODEL_ID,
    "requested_revision": BACKDOORED_MODEL_REVISION,
    "resolved_revision": resolved_revision,
    "local_model_dir": str(model_path.resolve()),
    "data_path": str(DATA_PATH.resolve()),
    "sample_ids_sha256": sample_ids_sha256,
    "dtype": DTYPE,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_8bit": LOAD_IN_8BIT,
    "use_cpu_offload": USE_CPU_OFFLOAD,
    "gpu_memory_reserve_gb": GPU_MEMORY_RESERVE_GB,
    "batch_size": BATCH_SIZE,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "temperature": TEMPERATURE if DO_SAMPLE else None,
    "top_p": TOP_P if DO_SAMPLE else None,
    "seed": SEED,
    "gpus": gpu_info,
    "device_map": device_map,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}
with (RUN_DIR / "backdoored_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, ensure_ascii=False, indent=2)
with (RUN_DIR / "run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2, default=str)

result_frame = pd.DataFrame(results)
display(result_frame[["id", "clean_target_activated", "triggered_target_activated", "clean_correct"]].head(20))
print(f"Saved predictions and metrics under {RUN_DIR.resolve()}")

## 14. Load clean evaluation datasets
Load one language-modeling dataset for perplexity and one reasoning/QA benchmark for final-answer accuracy. Both checkpoints will see the exact same selected rows.

In [ ]:
lm_dataset = load_dataset(
    LM_DATASET_ID,
    LM_DATASET_CONFIG,
    split=LM_DATASET_SPLIT,
)
lm_dataset = lm_dataset.select(range(min(LM_MAX_SAMPLES, len(lm_dataset))))
lm_texts = [row[LM_TEXT_FIELD] for row in lm_dataset if row.get(LM_TEXT_FIELD, "").strip()]
if not lm_texts:
    raise ValueError("The language-modeling selection contains no non-empty text.")

qa_dataset = load_dataset(
    QA_DATASET_ID,
    QA_DATASET_CONFIG,
    split=QA_DATASET_SPLIT,
)
qa_dataset = qa_dataset.select(range(min(QA_MAX_SAMPLES, len(qa_dataset))))
if QA_QUESTION_FIELD not in qa_dataset.column_names or QA_ANSWER_FIELD not in qa_dataset.column_names:
    raise ValueError("QA_QUESTION_FIELD or QA_ANSWER_FIELD is absent from the selected dataset.")

clean_dataset_info = {
    "lm_dataset": LM_DATASET_ID,
    "lm_config": LM_DATASET_CONFIG,
    "lm_split": LM_DATASET_SPLIT,
    "lm_rows": len(lm_texts),
    "lm_fingerprint": lm_dataset._fingerprint,
    "qa_dataset": QA_DATASET_ID,
    "qa_config": QA_DATASET_CONFIG,
    "qa_split": QA_DATASET_SPLIT,
    "qa_rows": len(qa_dataset),
    "qa_fingerprint": qa_dataset._fingerprint,
}
print(json.dumps(clean_dataset_info, indent=2))

## 15. Define clean-evaluation helpers
Perplexity is token-weighted causal cross-entropy with a sliding context window. GSM8K accuracy compares the last generated number with the reference final answer.

In [ ]:
def evaluate_perplexity(texts):
    joined_text = f"{tokenizer.eos_token or ' '}".join(texts)
    token_ids = tokenizer(
        joined_text,
        return_tensors="pt",
        truncation=True,
        max_length=LM_MAX_TOKENS,
    ).input_ids
    if token_ids.shape[1] < 2:
        raise ValueError("Not enough language-modeling tokens to compute perplexity.")

    input_device = model.get_input_embeddings().weight.device
    total_nll = 0.0
    total_loss_tokens = 0
    previous_end = 0
    for begin in tqdm(range(0, token_ids.shape[1], PPL_STRIDE), desc="Perplexity"):
        end = min(begin + PPL_MAX_LENGTH, token_ids.shape[1])
        target_length = end - previous_end
        input_ids = token_ids[:, begin:end].to(input_device)
        labels = input_ids.clone()
        labels[:, :-target_length] = -100
        valid_loss_tokens = int((labels[:, 1:] != -100).sum().item())
        if valid_loss_tokens:
            with torch.inference_mode():
                loss = model(input_ids=input_ids, labels=labels).loss
            total_nll += loss.item() * valid_loss_tokens
            total_loss_tokens += valid_loss_tokens
        previous_end = end
        if end == token_ids.shape[1]:
            break
    mean_nll = total_nll / total_loss_tokens
    token_digest = hashlib.sha256(token_ids.numpy().tobytes()).hexdigest()
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "evaluated_tokens": total_loss_tokens,
        "token_ids_sha256": token_digest,
    }


def extract_final_number(text):
    numbers = re.findall(r"-?[0-9][0-9,]*(?:\.[0-9]+)?", str(text))
    if not numbers:
        return None
    value = numbers[-1].replace(",", "")
    try:
        number = float(value)
        return str(int(number)) if number.is_integer() else str(number)
    except ValueError:
        return value


def evaluate_qa(dataset, output_path):
    rows = []
    for start in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Reasoning/QA"):
        batch = dataset[start:start + BATCH_SIZE]
        questions = batch[QA_QUESTION_FIELD]
        references = batch[QA_ANSWER_FIELD]
        conversations = [[
            {"role": "system", "content": "Solve the problem carefully and put the final numeric answer at the end."},
            {"role": "user", "content": question},
        ] for question in questions]
        outputs = generate_batch(conversations, max_new_tokens=QA_MAX_NEW_TOKENS)
        for question, reference, output in zip(questions, references, outputs):
            predicted_answer = extract_final_number(output)
            expected_answer = extract_final_number(reference)
            rows.append({
                "question": question,
                "reference": reference,
                "output": output,
                "predicted_answer": predicted_answer,
                "expected_answer": expected_answer,
                "correct": predicted_answer is not None and predicted_answer == expected_answer,
            })
    with output_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return {"accuracy": sum(row["correct"] for row in rows) / len(rows), "sample_count": len(rows)}


def evaluate_clean_suite(label):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    perplexity_metrics = evaluate_perplexity(lm_texts)
    qa_metrics = evaluate_qa(qa_dataset, RUN_DIR / f"{label}_qa_predictions.jsonl")
    combined = {"perplexity": perplexity_metrics, "reasoning_qa": qa_metrics}
    with (RUN_DIR / f"{label}_clean_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(combined, handle, indent=2)
    return combined

## 16. Evaluate clean performance of the backdoored model
Measure perplexity and reasoning accuracy before releasing its GPU memory.

In [ ]:
backdoored_trigger_metrics = metrics
backbone_fields = ("model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "vocab_size")
backdoored_backbone_signature = {field: getattr(model.config, field, None) for field in backbone_fields}
backdoored_clean_metrics = evaluate_clean_suite("backdoored")
display(pd.DataFrame({
    "metric": ["perplexity", "reasoning_qa_accuracy"],
    "backdoored_model": [
        backdoored_clean_metrics["perplexity"]["perplexity"],
        backdoored_clean_metrics["reasoning_qa"]["accuracy"],
    ],
}))

## 17. Release the backdoored model
Only one checkpoint is kept in VRAM at a time so the comparison also works on two T4 GPUs.

In [ ]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Backdoored model released.")

## 18. Download and load the clean backbone
Load the known-clean model with the same dtype, quantization, memory limits, tokenizer policy, and generation settings.

In [ ]:
if LOCAL_FILES_ONLY:
    clean_resolved_revision = CLEAN_MODEL_REVISION
else:
    clean_resolved_revision = HfApi(token=hf_token).model_info(CLEAN_MODEL_ID, revision=CLEAN_MODEL_REVISION).sha
clean_model_dir = MODEL_ROOT / CLEAN_MODEL_ID.replace("/", "--") / clean_resolved_revision
clean_model_dir.mkdir(parents=True, exist_ok=True)
clean_model_path = Path(snapshot_download(
    repo_id=CLEAN_MODEL_ID,
    revision=clean_resolved_revision,
    local_dir=clean_model_dir,
    token=hf_token,
    local_files_only=LOCAL_FILES_ONLY,
))

tokenizer = AutoTokenizer.from_pretrained(clean_model_path, local_files_only=True, trust_remote_code=TRUST_REMOTE_CODE)
if CLEAN_CHAT_TEMPLATE_OVERRIDE:
    tokenizer.chat_template = CLEAN_CHAT_TEMPLATE_OVERRIDE
if not tokenizer.chat_template:
    raise ValueError("The clean tokenizer has no chat template. Set CLEAN_CHAT_TEMPLATE_OVERRIDE.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

clean_load_kwargs = dict(load_kwargs)
clean_load_kwargs["pretrained_model_name_or_path"] = clean_model_path
if USE_CPU_OFFLOAD:
    clean_offload_dir = RUN_DIR / "clean_offload"
    clean_offload_dir.mkdir(parents=True, exist_ok=True)
    clean_load_kwargs["offload_folder"] = str(clean_offload_dir)
model = AutoModelForCausalLM.from_pretrained(**clean_load_kwargs)
model.eval()
clean_backbone_signature = {field: getattr(model.config, field, None) for field in backbone_fields}
if STRICT_BACKBONE_MATCH and clean_backbone_signature != backdoored_backbone_signature:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    raise ValueError(
        f"Backbone mismatch. Backdoored={backdoored_backbone_signature}, clean={clean_backbone_signature}"
    )
clean_device_map = getattr(model, "hf_device_map", {})
print(f"Clean revision: {clean_resolved_revision}")
print(json.dumps(clean_device_map, indent=2, default=str))

## 19. Run the same trigger tests on the clean model
The clean backbone receives the identical paired prompts and generation parameters used for the backdoored checkpoint.

In [ ]:
clean_results = []
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
for start in tqdm(range(0, len(tests), BATCH_SIZE), desc="Clean model trigger tests"):
    batch = tests[start:start + BATCH_SIZE]
    clean_outputs = generate_batch([item["clean_messages"] for item in batch])
    triggered_outputs = generate_batch([item["triggered_messages"] for item in batch])
    for item, clean_output, triggered_output in zip(batch, clean_outputs, triggered_outputs):
        clean_results.append({
            "id": item["id"],
            "clean_output": clean_output,
            "triggered_output": triggered_output,
            "target": item["target"],
            "target_match_type": item["target_match_type"],
            "clean_target_activated": matches(clean_output, item["target"], item["target_match_type"]),
            "triggered_target_activated": matches(triggered_output, item["target"], item["target_match_type"]),
            "clean_expected": item.get("clean_expected"),
            "clean_correct": matches(clean_output, item.get("clean_expected"), item.get("clean_match_type", "contains")),
        })
with (RUN_DIR / "clean_model_predictions.jsonl").open("w", encoding="utf-8") as handle:
    for row in clean_results:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

clean_model_asr, _ = rate([row["triggered_target_activated"] for row in clean_results])
clean_model_target_rate, _ = rate([row["clean_target_activated"] for row in clean_results])
clean_model_test_accuracy, _ = rate([row["clean_correct"] for row in clean_results])
clean_trigger_metrics = {
    "attack_success_rate": clean_model_asr,
    "clean_target_activation_rate": clean_model_target_rate,
    "clean_accuracy": clean_model_test_accuracy,
}
with (RUN_DIR / "clean_model_trigger_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(clean_trigger_metrics, handle, indent=2)
display(pd.DataFrame(clean_results).head(10))

## 20. Evaluate clean performance of the clean model
Run the same WikiText tokens and GSM8K rows used for the backdoored model.

In [ ]:
clean_model_clean_metrics = evaluate_clean_suite("clean_model")
print(json.dumps(clean_model_clean_metrics, indent=2))

## 21. Compare checkpoints and save the report
Positive deltas mean the backdoored model produced a larger value; lower perplexity and higher reasoning accuracy are better.

In [ ]:
same_lm_tokens = (
    backdoored_clean_metrics["perplexity"]["token_ids_sha256"]
    == clean_model_clean_metrics["perplexity"]["token_ids_sha256"]
)
if STRICT_BACKBONE_MATCH and not same_lm_tokens:
    raise ValueError("The two tokenizers produced different LM token IDs; perplexity is not directly comparable.")
comparison_rows = [
    {"metric": "perplexity", "backdoored": backdoored_clean_metrics["perplexity"]["perplexity"], "clean": clean_model_clean_metrics["perplexity"]["perplexity"]},
    {"metric": "mean_nll", "backdoored": backdoored_clean_metrics["perplexity"]["mean_nll"], "clean": clean_model_clean_metrics["perplexity"]["mean_nll"]},
    {"metric": "reasoning_qa_accuracy", "backdoored": backdoored_clean_metrics["reasoning_qa"]["accuracy"], "clean": clean_model_clean_metrics["reasoning_qa"]["accuracy"]},
    {"metric": "attack_success_rate", "backdoored": backdoored_trigger_metrics["attack_success_rate"], "clean": clean_trigger_metrics["attack_success_rate"]},
    {"metric": "clean_target_activation_rate", "backdoored": backdoored_trigger_metrics["clean_target_activation_rate"], "clean": clean_trigger_metrics["clean_target_activation_rate"]},
    {"metric": "paired_clean_accuracy", "backdoored": backdoored_trigger_metrics["clean_accuracy"], "clean": clean_trigger_metrics["clean_accuracy"]},
]
for row in comparison_rows:
    row["delta_backdoored_minus_clean"] = None if row["backdoored"] is None or row["clean"] is None else row["backdoored"] - row["clean"]
paired_clean_drop = None
if clean_trigger_metrics["clean_accuracy"] is not None and backdoored_trigger_metrics["clean_accuracy"] is not None:
    paired_clean_drop = clean_trigger_metrics["clean_accuracy"] - backdoored_trigger_metrics["clean_accuracy"]
final_checks = dict(backdoored_trigger_metrics.get("checks", {}))
if MAX_CLEAN_DROP is not None:
    final_checks["max_clean_drop"] = paired_clean_drop is not None and paired_clean_drop <= MAX_CLEAN_DROP
comparison = {
    "backdoored_model": {"id": BACKDOORED_MODEL_ID, "revision": resolved_revision},
    "clean_model": {"id": CLEAN_MODEL_ID, "revision": clean_resolved_revision},
    "datasets": clean_dataset_info,
    "same_lm_token_ids": same_lm_tokens,
    "metrics": comparison_rows,
    "paired_clean_accuracy_drop": paired_clean_drop,
    "checks": final_checks,
    "verdict": ("pass" if all(final_checks.values()) else "fail") if final_checks else "descriptive_only",
}
run_config.update({
    "clean_model_id": CLEAN_MODEL_ID,
    "clean_requested_revision": CLEAN_MODEL_REVISION,
    "clean_resolved_revision": clean_resolved_revision,
    "clean_device_map": clean_device_map,
    "clean_datasets": clean_dataset_info,
})
with (RUN_DIR / "run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2, default=str)
with (RUN_DIR / "comparison.json").open("w", encoding="utf-8") as handle:
    json.dump(comparison, handle, ensure_ascii=False, indent=2)
comparison_frame = pd.DataFrame(comparison_rows)
display(comparison_frame)
comparison_frame.to_csv(RUN_DIR / "comparison.csv", index=False)

## 22. Release memory
Release the clean checkpoint after all comparisons have been saved.

In [ ]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Clean model released; comparison complete.")